# ☕ The Coffee Crisis
### A Data Analysis of Coffee Prices, Climate Change, and the Future of Your Morning Cup

---

**Author:** Alejandro Molina  
**Created:** 2026  
**Tools:** Python 3.12, Pandas, Scikit-learn, yfinance, NASA GISTEMP API, World Bank Data360 API, FAO Data, Tableau Public  
**Assistant:** Claude AI (Anthropic) — guidance & debugging

**Data Sources:**
- Coffee Prices: Macrotrends / ICO Historical Commodity Data
- Temperature: NASA GISTEMP v4 — Goddard Institute for Space Studies
- Production: UN Food and Agriculture Organization (FAO)
- Yields: Our World in Data / FAO
- Inflation/CPI: US Bureau of Labor Statistics

**Description:**  
This project investigates the relationship between rising coffee prices, climate change, 
and future supply collapse. It began with a simple observation at Walmart — a $1.20 price 
increase on a favorite brand — and grew into a full data pipeline connecting climate science, 
agricultural data, and commodity markets.

**Output Files:**
- `coffee_master_dataset.csv` — 51 years of combined data (1973–2023)
- `coffee_price_forecast.csv` — Price projections to 2040 (3 scenarios)
- `temperature_forecast.csv` — Temperature projections to 2040 (3 scenarios)
- `yield_forecast.csv` — Yield projections to 2040 (3 scenarios)

---

## Cell 1 — Setup & Imports
Always run this cell first. It fixes SSL certificate issues on Windows/Anaconda
and imports all required libraries.

In [1]:
# ── SSL Certificate Fix (required for Anaconda on Windows) ────────
import certifi
import os
os.environ["SSL_CERT_FILE"] = certifi.where()

# ── Core imports ──────────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

print("✅ Ready to go!")

✅ Ready to go!


## Cell 2 — Load Coffee Prices
Source: Macrotrends / ICO Historical Commodity Data  
File: `coffee-prices-historical-data.csv`  
Coverage: Daily prices 1973–2023 → aggregated to annual averages

In [2]:
# ── Load raw daily coffee prices ─────────────────────────────────
df_prices = pd.read_csv("coffee-prices-historical-data.csv")
df_prices.columns = ["date", "price_usd_per_lb"]

# Handle mixed date formats (YYYY-MM-DD and DD.MM.YYYY)
df_prices["date"] = pd.to_datetime(df_prices["date"], format="mixed", dayfirst=True)
df_prices["year"] = df_prices["date"].dt.year

# Annual average price per year
df_prices_annual = df_prices.groupby("year")["price_usd_per_lb"].mean().reset_index()

print(f"✅ Coffee Prices: {len(df_prices_annual)} years of data")
print(f"   Years: {df_prices_annual['year'].min()} to {df_prices_annual['year'].max()}")
print(f"   Price range: ${df_prices_annual['price_usd_per_lb'].min():.2f} to ${df_prices_annual['price_usd_per_lb'].max():.2f} per lb")
print(df_prices_annual.tail(5))

✅ Coffee Prices: 51 years of data
   Years: 1973 to 2023
   Price range: $0.54 to $2.53 per lb
    year  price_usd_per_lb
46  2019          1.018240
47  2020          1.113771
48  2021          1.693321
49  2022          2.144288
50  2023          1.740848


## Cell 3 — Load Temperature Data
Source: NASA GISTEMP v4 — Goddard Institute for Space Studies  
URL: https://data.giss.nasa.gov/gistemp/  
Coverage: Annual global mean temperature anomalies relative to 1951–1980 baseline  
Note: +1.0°C is the coffee plant stress threshold

In [3]:
# ── Pull NASA GISTEMP data directly ──────────────────────────────
url = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"
df_temps = pd.read_csv(url, skiprows=1, na_values="***")

# Keep year and annual average (J-D = January to December mean)
df_temps = df_temps[["Year", "J-D"]].rename(columns={"J-D": "temp_anomaly_c"})
df_temps = df_temps.dropna()

# Add stress threshold flag
df_temps["above_stress_threshold"] = df_temps["temp_anomaly_c"] >= 1.0

# Save
df_temps.to_csv("temperature_data.csv", index=False)

print(f"✅ Temperature Data: {len(df_temps)} years of data")
print(f"   Years: {df_temps['Year'].min()} to {df_temps['Year'].max()}")
print(f"   Years above stress threshold: {df_temps['above_stress_threshold'].sum()}")
print(f"\nAverage temp anomaly 2000-2010: {df_temps[df_temps['Year'].between(2000,2010)]['temp_anomaly_c'].mean():.3f}°C")
print(f"Average temp anomaly 2015-2025: {df_temps[df_temps['Year'].between(2015,2025)]['temp_anomaly_c'].mean():.3f}°C")

✅ Temperature Data: 146 years of data
   Years: 1880 to 2025
   Years above stress threshold: 5

Average temp anomaly 2000-2010: 0.597°C
Average temp anomaly 2015-2025: 1.003°C


## Cell 4 — Load Coffee Production Data
Source: UN Food and Agriculture Organization (FAO)  
File: `coffee-bean-production.csv`  
Coverage: Annual production in tonnes for top 5 coffee countries (1961–2024)  
Countries: Brazil (BRA), Vietnam (VNM), Colombia (COL), Ethiopia (ETH), Indonesia (IDN)

In [4]:
# ── Load FAO production data ──────────────────────────────────────
df_prod = pd.read_csv("coffee-bean-production.csv")
df_prod.columns = ["country", "country_code", "year", "production_tonnes"]

# Filter for top 5 coffee producing countries
coffee_countries = ["BRA", "VNM", "COL", "ETH", "IDN"]
df_prod_filtered = df_prod[df_prod["country_code"].isin(coffee_countries)]

# Total combined production per year across all 5 countries
df_prod_annual = df_prod_filtered.groupby("year")["production_tonnes"] \
    .sum().reset_index()

print(f"✅ Production Data: {len(df_prod_annual)} years of data")
print(f"   Years: {df_prod_annual['year'].min()} to {df_prod_annual['year'].max()}")
print(f"   Countries: {', '.join(coffee_countries)}")
print(df_prod_annual.tail(5))

✅ Production Data: 64 years of data
   Years: 1961 to 2024
   Countries: BRA, VNM, COL, ETH, IDN
    year  production_tonnes
59  2020         7649764.56
60  2021         6767545.00
61  2022         7118873.94
62  2023         7304275.20
63  2024         7626133.90


## Cell 5 — Load Coffee Yield Data
Source: Our World in Data / FAO  
File: `coffee-yields.csv`  
Coverage: Annual yield in tonnes per hectare (1961–2024)  
Note: Declining yields indicate climate stress on farming

In [5]:
# ── Load yield data ───────────────────────────────────────────────
df_yields = pd.read_csv("coffee-yields.csv")
df_yields.columns = ["country", "country_code", "year", "yield_tonnes_per_hectare"]

# Filter for our 5 key countries
df_yields_filtered = df_yields[df_yields["country_code"].isin(coffee_countries)].copy()

# Average yield across all 5 countries per year
df_yields_annual = df_yields_filtered.groupby("year")["yield_tonnes_per_hectare"] \
    .mean().reset_index()

print(f"✅ Yield Data: {len(df_yields_annual)} years of data")
print(f"   Years: {df_yields_annual['year'].min()} to {df_yields_annual['year'].max()}")
print(df_yields_annual.tail(5))

✅ Yield Data: 64 years of data
   Years: 1961 to 2024
    year  yield_tonnes_per_hectare
59  2020                   1.39940
60  2021                   1.27648
61  2022                   1.35400
62  2023                   1.35252
63  2024                   1.40370


## Cell 6 — Load CPI & Inflation Data
Source: US Bureau of Labor Statistics  
File: `cpi_in_usa.csv`  
Coverage: Annual US Consumer Price Index and inflation rate (1973–2023)  
Used to calculate inflation-adjusted coffee prices

In [6]:
# ── Load CPI data ─────────────────────────────────────────────────
df_cpi = pd.read_csv("cpi_in_usa.csv", sep=";")
df_cpi.columns = ["year", "cpi", "inflation_rate"]

# Clean up inflation rate — remove % sign and convert to float
df_cpi["inflation_rate"] = df_cpi["inflation_rate"].str.replace("%", "").astype(float)
df_cpi["year"] = df_cpi["year"].astype(int)

print(f"✅ CPI Data: {len(df_cpi)} years of data")
print(f"   Years: {df_cpi['year'].min()} to {df_cpi['year'].max()}")
print(df_cpi.tail(5))

✅ CPI Data: 51 years of data
   Years: 1973 to 2023
    year    cpi  inflation_rate
46  2019  255.7             1.8
47  2020  258.8             1.2
48  2021  271.0             4.7
49  2022  292.7             8.0
50  2023  304.3             4.0


## Cell 7 — Build Master Dataset
Merges all 5 datasets on year into one clean combined table.  
Also calculates inflation-adjusted coffee price (real 2000 dollars).  
Output: `coffee_master_dataset.csv`

In [7]:
# ── Merge all datasets on year ────────────────────────────────────
df_master = df_prices_annual.merge(df_temps, left_on="year", right_on="Year")

# Clean up duplicate year column from temperature data
if "Year" in df_master.columns:
    df_master = df_master.drop(columns=["Year"])
if "above_stress_threshold" in df_master.columns:
    df_master = df_master.drop(columns=["above_stress_threshold"])

# Add production
df_master = df_master.merge(df_prod_annual, on="year", how="left")

# Add CPI and inflation
df_master["year"] = df_master["year"].astype(int)
df_master = df_master.merge(df_cpi[["year", "cpi", "inflation_rate"]], on="year", how="left")

# Add yields
df_master = df_master.merge(df_yields_annual, on="year", how="left")

# Calculate inflation-adjusted price (real 2000 dollars)
base_cpi = df_cpi[df_cpi["year"] == 2000]["cpi"].iloc[0]
df_master["real_price_2000_dollars"] = (df_master["price_usd_per_lb"] * (base_cpi / df_master["cpi"])).round(4)

# Remove rows with nulls in key columns
df_master = df_master.dropna(subset=[
    "price_usd_per_lb",
    "temp_anomaly_c",
    "production_tonnes",
    "cpi",
    "inflation_rate",
    "real_price_2000_dollars"
])

# Save
df_master.to_csv("coffee_master_dataset.csv", index=False)

print("✅ MASTER DATASET COMPLETE")
print(f"   Rows: {len(df_master)}")
print(f"   Years: {df_master['year'].min()} to {df_master['year'].max()}")
print(f"   Columns: {df_master.columns.tolist()}")
print(f"\n   Price range:      ${df_master['price_usd_per_lb'].min():.2f} to ${df_master['price_usd_per_lb'].max():.2f} per lb")
print(f"   Real price range: ${df_master['real_price_2000_dollars'].min():.2f} to ${df_master['real_price_2000_dollars'].max():.2f} (2000 dollars)")
print(f"   Temp range:       {df_master['temp_anomaly_c'].min()}°C to {df_master['temp_anomaly_c'].max()}°C")
print(f"   Production range: {df_master['production_tonnes'].min():,.0f} to {df_master['production_tonnes'].max():,.0f} tonnes")
print(f"\n   Saved: coffee_master_dataset.csv")

✅ MASTER DATASET COMPLETE
   Rows: 51
   Years: 1973 to 2023
   Columns: ['year', 'price_usd_per_lb', 'temp_anomaly_c', 'production_tonnes', 'cpi', 'inflation_rate', 'yield_tonnes_per_hectare', 'real_price_2000_dollars']

   Price range:      $0.54 to $2.53 per lb
   Real price range: $0.52 to $6.58 (2000 dollars)
   Temp range:       -0.1°C to 1.17°C
   Production range: 1,062,062 to 7,649,765 tonnes

   Saved: coffee_master_dataset.csv


## Cell 8 — Price Forecast to 2040
Generates three price scenarios based on IPCC climate projections and 
ICO/World Coffee Research analyst estimates.  
Uses numpy interpolation between research-based anchor points.  
Output: `coffee_price_forecast.csv`

In [8]:
# ── Research-based price anchors (USD per lb) ─────────────────────
# Sources: ICO, World Coffee Research, analyst projections
manual_forecasts = {
    "Optimistic": {
        # Emissions reduced, good harvests, prices stabilize
        2024: 1.80, 2025: 1.90, 2026: 1.85, 2027: 1.90,
        2028: 1.95, 2029: 2.00, 2030: 2.05, 2032: 2.10,
        2035: 2.20, 2037: 2.25, 2040: 2.30
    },
    "Mid": {
        # Current trajectory continues
        2024: 1.90, 2025: 2.10, 2026: 2.00, 2027: 2.20,
        2028: 2.40, 2029: 2.50, 2030: 2.70, 2032: 2.90,
        2035: 3.20, 2037: 3.50, 2040: 3.80
    },
    "Pessimistic": {
        # Climate disruption accelerates, supply collapses
        2024: 2.00, 2025: 2.50, 2026: 2.80, 2027: 3.20,
        2028: 3.60, 2029: 4.00, 2030: 4.50, 2032: 5.20,
        2035: 6.50, 2037: 7.80, 2040: 9.00
    }
}

records = []

# Include historical actuals for chart continuity
for _, row in df_master.iterrows():
    records.append({
        "year":             int(row["year"]),
        "price_usd_per_lb": round(row["price_usd_per_lb"], 4),
        "scenario":         "Historical",
        "is_forecast":      False
    })

# Interpolate between anchor points for smooth forecast lines
for scenario, anchors in manual_forecasts.items():
    years      = sorted(anchors.keys())
    prices     = [anchors[y] for y in years]
    all_years  = list(range(2024, 2041))
    all_prices = np.interp(all_years, years, prices)

    for year, price in zip(all_years, all_prices):
        records.append({
            "year":             year,
            "price_usd_per_lb": round(price, 4),
            "scenario":         scenario,
            "is_forecast":      True
        })

df_forecast = pd.DataFrame(records)
df_forecast.to_csv("coffee_price_forecast.csv", index=False)

print(f"✅ Price Forecast: {len(df_forecast)} rows saved")
print(f"\n2040 price predictions:")
for scenario in ["Optimistic", "Mid", "Pessimistic"]:
    price = df_forecast[
        (df_forecast["scenario"] == scenario) &
        (df_forecast["year"] == 2040)
    ]["price_usd_per_lb"].iloc[0]
    print(f"   {scenario}: ${price:.2f}/lb")
print(f"\n   Saved: coffee_price_forecast.csv")

✅ Price Forecast: 102 rows saved

2040 price predictions:
   Optimistic: $2.30/lb
   Mid: $3.80/lb
   Pessimistic: $9.00/lb

   Saved: coffee_price_forecast.csv


## Cell 9 — Temperature Forecast to 2040
Generates three temperature scenarios aligned with IPCC AR6 pathways.  
All scenarios remain above the +1.0°C coffee stress threshold.  
Output: `temperature_forecast.csv`

In [9]:
# ── Temperature projections (°C anomaly vs 1951-1980 baseline) ────
# Source: IPCC AR6, NASA projections
temp_forecasts = {
    "Optimistic": {
        # Strong emissions cuts — warming slows near Paris target
        2024: 1.20, 2025: 1.22, 2026: 1.23, 2027: 1.24,
        2028: 1.25, 2029: 1.26, 2030: 1.27, 2032: 1.28,
        2035: 1.30, 2037: 1.32, 2040: 1.35
    },
    "Mid": {
        # Current policies continue
        2024: 1.22, 2025: 1.25, 2026: 1.28, 2027: 1.31,
        2028: 1.35, 2029: 1.38, 2030: 1.42, 2032: 1.50,
        2035: 1.62, 2037: 1.72, 2040: 1.85
    },
    "Pessimistic": {
        # No major climate action — accelerating warming
        2024: 1.25, 2025: 1.30, 2026: 1.36, 2027: 1.42,
        2028: 1.49, 2029: 1.56, 2030: 1.64, 2032: 1.81,
        2035: 2.05, 2037: 2.25, 2040: 2.55
    }
}

temp_records = []

# Include historical actuals for chart continuity
for _, row in df_master.iterrows():
    if pd.notna(row["temp_anomaly_c"]):
        temp_records.append({
            "year":           int(row["year"]),
            "temp_anomaly_c": round(row["temp_anomaly_c"], 4),
            "scenario":       "Historical",
            "is_forecast":    False
        })

# Interpolate between anchor points
for scenario, anchors in temp_forecasts.items():
    years     = sorted(anchors.keys())
    temps     = [anchors[y] for y in years]
    all_years = list(range(2024, 2041))
    all_temps = np.interp(all_years, years, temps)

    for year, temp in zip(all_years, all_temps):
        temp_records.append({
            "year":           year,
            "temp_anomaly_c": round(temp, 4),
            "scenario":       scenario,
            "is_forecast":    True
        })

df_temp_forecast = pd.DataFrame(temp_records)
df_temp_forecast.to_csv("temperature_forecast.csv", index=False)

print(f"✅ Temperature Forecast: {len(df_temp_forecast)} rows saved")
print(f"\n2040 temperature predictions:")
for scenario in ["Optimistic", "Mid", "Pessimistic"]:
    temp = df_temp_forecast[
        (df_temp_forecast["scenario"] == scenario) &
        (df_temp_forecast["year"] == 2040)
    ]["temp_anomaly_c"].iloc[0]
    print(f"   {scenario}: +{temp}°C")
print(f"\n   Note: All scenarios remain above +1.0°C coffee stress threshold")
print(f"   Saved: temperature_forecast.csv")

✅ Temperature Forecast: 102 rows saved

2040 temperature predictions:
   Optimistic: +1.35°C
   Mid: +1.85°C
   Pessimistic: +2.55°C

   Note: All scenarios remain above +1.0°C coffee stress threshold
   Saved: temperature_forecast.csv


## Cell 10 — Yield Forecast to 2040
Generates three yield scenarios showing how climate change reduces
coffee output per hectare over time.  
Source: World Coffee Research / IPCC agricultural projections  
Output: `yield_forecast.csv`

In [10]:
# ── Yield projections (tonnes per hectare) ────────────────────────
# Higher temps = lower yields — current avg is ~1.35 t/ha
yield_forecasts = {
    "Optimistic": {
        # Slow decline — climate adaptation and new varieties help
        2024: 1.40, 2025: 1.39, 2026: 1.38, 2027: 1.37,
        2028: 1.36, 2029: 1.35, 2030: 1.34, 2032: 1.32,
        2035: 1.28, 2037: 1.25, 2040: 1.20
    },
    "Mid": {
        # Steady decline as temperatures rise
        2024: 1.38, 2025: 1.35, 2026: 1.31, 2027: 1.27,
        2028: 1.23, 2029: 1.19, 2030: 1.15, 2032: 1.08,
        2035: 0.98, 2037: 0.90, 2040: 0.80
    },
    "Pessimistic": {
        # Sharp collapse — climate disruption accelerates
        2024: 1.35, 2025: 1.28, 2026: 1.20, 2027: 1.11,
        2028: 1.02, 2029: 0.93, 2030: 0.84, 2032: 0.68,
        2035: 0.48, 2037: 0.32, 2040: 0.18
    }
}

yield_records = []

# Include historical actuals for chart continuity
for _, row in df_master.iterrows():
    if pd.notna(row["yield_tonnes_per_hectare"]):
        yield_records.append({
            "year":                     int(row["year"]),
            "yield_tonnes_per_hectare": round(row["yield_tonnes_per_hectare"], 4),
            "scenario":                 "Historical",
            "is_forecast":              False
        })

# Interpolate between anchor points
for scenario, anchors in yield_forecasts.items():
    years      = sorted(anchors.keys())
    yields     = [anchors[y] for y in years]
    all_years  = list(range(2024, 2041))
    all_yields = np.interp(all_years, years, yields)

    for year, yld in zip(all_years, all_yields):
        yield_records.append({
            "year":                     year,
            "yield_tonnes_per_hectare": round(yld, 4),
            "scenario":                 scenario,
            "is_forecast":              True
        })

df_yield_forecast = pd.DataFrame(yield_records)
df_yield_forecast.to_csv("yield_forecast.csv", index=False)

print(f"✅ Yield Forecast: {len(df_yield_forecast)} rows saved")
print(f"\n2040 yield predictions (tonnes per hectare):")
for scenario in ["Optimistic", "Mid", "Pessimistic"]:
    yld = df_yield_forecast[
        (df_yield_forecast["scenario"] == scenario) &
        (df_yield_forecast["year"] == 2040)
    ]["yield_tonnes_per_hectare"].iloc[0]
    print(f"   {scenario}: {yld} t/ha")
print(f"\n   Current (2023): ~1.35 t/ha")
print(f"   Saved: yield_forecast.csv")

✅ Yield Forecast: 102 rows saved

2040 yield predictions (tonnes per hectare):
   Optimistic: 1.2 t/ha
   Mid: 0.8 t/ha
   Pessimistic: 0.18 t/ha

   Current (2023): ~1.35 t/ha
   Saved: yield_forecast.csv


## Cell 11 — Final Summary
Verifies all output files and prints key findings.

In [11]:
# ── Verify all output files ───────────────────────────────────────
output_files = [
    "coffee_master_dataset.csv",
    "coffee_price_forecast.csv",
    "temperature_forecast.csv",
    "yield_forecast.csv",
    "temperature_data.csv"
]

print("=" * 55)
print("  THE COFFEE CRISIS — Output Files")
print("=" * 55)
for f in output_files:
    if os.path.exists(f):
        df_check = pd.read_csv(f)
        print(f"✅ {f}")
        print(f"   {len(df_check)} rows, {len(df_check.columns)} columns")
    else:
        print(f"❌ {f} — NOT FOUND")

print()
print("=" * 55)
print("  KEY FINDINGS")
print("=" * 55)
print(f"  Price 1973:         ${df_master['price_usd_per_lb'].iloc[0]:.2f}/lb")
print(f"  Price 2023:         ${df_master['price_usd_per_lb'].iloc[-1]:.2f}/lb")
print(f"  Price increase:     {((df_master['price_usd_per_lb'].iloc[-1] / df_master['price_usd_per_lb'].iloc[0]) - 1) * 100:.0f}%")
print(f"  Temp 1973:          {df_master['temp_anomaly_c'].iloc[0]}°C anomaly")
print(f"  Temp 2023:          {df_master['temp_anomaly_c'].iloc[-1]}°C anomaly")
print(f"  Stress threshold:   +1.0°C (crossed in 2016)")
print(f"  Land lost by 2050:  ~50% (IPCC projection)")
print(f"  2040 price (Mid):   $3.80/lb")
print(f"  2040 price (Worst): $9.00/lb")
print("=" * 55)
print("  Load output CSVs into Tableau to build dashboard")
print("=" * 55)

  THE COFFEE CRISIS — Output Files
✅ coffee_master_dataset.csv
   51 rows, 8 columns
✅ coffee_price_forecast.csv
   102 rows, 4 columns
✅ temperature_forecast.csv
   102 rows, 4 columns
✅ yield_forecast.csv
   102 rows, 4 columns
✅ temperature_data.csv
   146 rows, 3 columns

  KEY FINDINGS
  Price 1973:         $0.66/lb
  Price 2023:         $1.74/lb
  Price increase:     165%
  Temp 1973:          0.16°C anomaly
  Temp 2023:          1.17°C anomaly
  Stress threshold:   +1.0°C (crossed in 2016)
  Land lost by 2050:  ~50% (IPCC projection)
  2040 price (Mid):   $3.80/lb
  2040 price (Worst): $9.00/lb
  Load output CSVs into Tableau to build dashboard
